# Request Scheduling and Continuous Batching

In our journey to understand vLLM, we've seen how the `Executor` runs the model, but how does it decide *what* to run? This notebook dives into the "brain" of the operation: the `Scheduler`. By the end of this notebook, you will understand the concept of continuous batching, which is key to vLLM's high throughput, and you will implement a simplified scheduler that manages multiple requests and their memory.

In [ ]:
# === Setup Cell ===
import collections
from dataclasses import dataclass, field
from enum import Enum
import random
import time
from IPython.display import display, clear_output

# --- Data Structures for Sequences ---

class SequenceStatus(Enum):
    """Represents the state of a sequence in the engine."""
    WAITING = 1
    RUNNING = 2
    FINISHED = 3

@dataclass
class Sequence:
    """Represents a single request's state."""
    seq_id: str
    prompt_tokens: list[int]
    output_tokens: list[int] = field(default_factory=list)
    status: SequenceStatus = SequenceStatus.WAITING
    # We will track allocated KV cache blocks here
    kv_blocks: list[int] = field(default_factory=list)

    def __len__(self) -> int:
        return len(self.prompt_tokens) + len(self.output_tokens)

    def __repr__(self):
        return f"Seq(id={self.seq_id}, status={self.status.name}, len={self.__len__()})"

### The Core Idea: The World's Most Efficient Restaurant

Imagine you're running a high-tech restaurant kitchen. Your most valuable asset is a set of super-advanced cooking stations (the GPU). You want to keep these stations busy 100% of the time to serve as many customers as possible.

**The Inefficient "Static Batching" Kitchen**

One approach is to wait until you have a full table of, say, four customers. You take all their orders, cook all four appetizers, serve them, then cook all four main courses, and so on. This is **static batching**.

What's the problem?
1.  **Idle Time:** If only three customers are waiting, the fourth station sits idle until another customer arrives.
2.  **Slowest Customer Wins:** If one customer is a very slow eater, the other three, who have finished their appetizers, must wait. Their plates can't be cleared, and their main courses can't be started until the slow eater is done. The entire table (the "batch") is blocked.

This is how traditional LLM inference works. It pads requests to the same length and waits for the entire batch to finish generating before starting the next one. The GPU is often idle, waiting for the slowest generation in the batch.

**The "Continuous Batching" Revolution**

A smarter chef would operate differently. The moment a customer finishes their appetizer, the chef immediately starts cooking their main course. As soon as one customer leaves, a new customer from the waiting line is seated. The kitchen operates at the level of individual dishes (tokens), not entire tables (requests).

This is **continuous batching**.

At every single step (every time a token is generated for *anyone*), the scheduler makes decisions:
1.  **Check for Finished Diners:** Has any request finished generating its sequence? If so, its cooking station and table space (KV cache) are immediately freed.
2.  **Admit New Diners:** Is there a free station and an open table? Great! Let's seat a new customer from the waiting queue and start cooking their first dish.

This approach ensures the cooking stations are always active, dramatically increasing the number of customers served per hour (i.e., the LLM's throughput). The scheduler is the master chef making these split-second decisions.

In [ ]:
# === Minimal Implementation ===
# Let's mock a simple Key-Value cache manager.
# Its only job is to track a fixed number of memory blocks.
class KVCacheManager:
    """A mock KV Cache that allocates memory blocks for sequences."""
    def __init__(self, num_total_blocks: int):
        # A pool of available block IDs
        self.free_blocks = list(range(num_total_blocks))

    def allocate(self, num_blocks_needed: int) -> list[int] | None:
        """Tries to allocate a certain number of blocks."""
        if len(self.free_blocks) >= num_blocks_needed:
            allocated = self.free_blocks[:num_blocks_needed]
            self.free_blocks = self.free_blocks[num_blocks_needed:]
            return allocated
        return None # Not enough memory

    def free(self, block_ids: list[int]):
        """Returns blocks to the free pool."""
        self.free_blocks.extend(block_ids)

class Scheduler:
    """The core scheduler that implements continuous batching."""
    def __init__(self, kv_cache_manager: KVCacheManager):
        self.kv_cache = kv_cache_manager
        self.waiting_queue = collections.deque()
        self.running_sequences = {} # Maps seq_id to Sequence object

    def add_sequence(self, seq: Sequence):
        """Adds a new request to the waiting queue."""
        self.waiting_queue.append(seq)

    def schedule(self) -> list[Sequence]:
        """The main scheduling logic for one iteration."""
        # 1. Free resources from sequences that finished in the last step.
        finished_ids = [
            seq_id for seq_id, seq in self.running_sequences.items()
            if seq.status == SequenceStatus.FINISHED
        ]
        for seq_id in finished_ids:
            seq = self.running_sequences.pop(seq_id)
            self.kv_cache.free(seq.kv_blocks)

        # 2. Try to schedule new sequences from the waiting queue.
        # We check the waiting queue as long as there's someone waiting
        # AND we have successfully scheduled the previous one.
        while self.waiting_queue:
            seq = self.waiting_queue[0] # Peek at the next sequence
            # For simplicity, let's say we need 1 KV block per 10 tokens.
            num_blocks_needed = (len(seq) + 9) // 10
            
            allocated_blocks = self.kv_cache.allocate(num_blocks_needed)
            if allocated_blocks:
                # Success! Move from waiting to running.
                seq = self.waiting_queue.popleft()
                seq.status = SequenceStatus.RUNNING
                seq.kv_blocks = allocated_blocks
                self.running_sequences[seq.seq_id] = seq
            else:
                # Not enough memory, we can't schedule any more.
                break
        
        # 3. Return the batch of all sequences that are currently running.
        return list(self.running_sequences.values())

### Walking Through the Implementation

Let's break down our simple `Scheduler`.

1.  **`KVCacheManager`**: This is our stand-in for vLLM's powerful PagedAttention memory manager. It has a simple job: maintain a list of `free_blocks`.
    *   `allocate(num_blocks)`: If it has enough blocks, it hands them over and removes them from the free list. Otherwise, it returns `None`, signaling that memory is full.
    *   `free(block_ids)`: When a sequence is finished, this method takes its blocks back and adds them to the `free_blocks` list.

2.  **`Scheduler`**: This is the chef.
    *   **`__init__`**: It holds the `kv_cache`, a `waiting_queue` for incoming requests (customers waiting for a table), and a `running_sequences` dictionary for requests currently being processed (customers who are seated and eating).
    *   **`add_sequence`**: A new request arrives and is added to the back of the line.
    *   **`schedule()`**: This is the heart of the continuous batching loop, executed at every single step.
        *   **Step 1: Free Resources (Clear the tables)**: It first checks the `running_sequences` for any that have been marked `FINISHED`. For each one, it tells the `KVCacheManager` to `free` its blocks and removes it from the running pool. This is crucial—it makes resources available for new requests.
        *   **Step 2: Schedule New Sequences (Seat new customers)**: It then looks at the `waiting_queue`. For the sequence at the front of the line, it calculates how many KV blocks it needs and asks the `KVCacheManager` to `allocate` them.
            *   If `allocate` succeeds, the sequence is moved from `waiting` to `running`.
            *   If `allocate` returns `None`, it means there's no memory. The scheduler stops trying to add new sequences for this iteration and `break`s the loop.
        *   **Step 3: Form the Batch (Tell the cooks what to make)**: Finally, it gathers all sequences currently in the `running_sequences` dictionary—both those that were already running and those that were just added—and returns them as the "batch" for the `Executor` to process in this step.

In [ ]:
# === Demonstration ===
# Let's watch the scheduler in action.

# Setup our environment
# We have a small KV cache with only 5 blocks.
kv_cache = KVCacheManager(num_total_blocks=5)
scheduler = Scheduler(kv_cache)
MAX_TOKENS_PER_SEQ = 30 # Stop generating after this many tokens.

# Create some initial requests
requests = [
    Sequence(seq_id="req_A", prompt_tokens=[1]*25), # Needs 3 blocks
    Sequence(seq_id="req_B", prompt_tokens=[2]*15), # Needs 2 blocks
    Sequence(seq_id="req_C", prompt_tokens=[3]*31), # Needs 4 blocks
    Sequence(seq_id="req_D", prompt_tokens=[4]*8),  # Needs 1 block
]

for req in requests:
    scheduler.add_sequence(req)

print("--- Starting Inference Simulation ---")
print(f"Initial state: {len(kv_cache.free_blocks)} free blocks. Waiting: {list(scheduler.waiting_queue)}\n")

# Simulate the engine's main loop for 10 steps
for step in range(10):
    # 1. The scheduler creates the batch for this step.
    running_batch = scheduler.schedule()

    # If there's nothing to run and nothing waiting, we're done.
    if not running_batch and not scheduler.waiting_queue:
        print("--- All requests finished! ---")
        break

    print(f"--- Step {step+1} ---")
    print(f"Scheduler created a batch of size: {len(running_batch)}")
    print(f"  Running: {[seq.seq_id for seq in running_batch]}")
    print(f"  Waiting: {[seq.seq_id for seq in scheduler.waiting_queue]}")
    print(f"  KV Cache Free Blocks: {len(kv_cache.free_blocks)}/{kv_cache.free_blocks.__len__() + sum(len(s.kv_blocks) for s in running_batch)}")

    # 2. The (mock) Executor processes the batch.
    # We'll just add one new token to each running sequence.
    if not running_batch:
        print("  Executor is idle this step.\n")
        continue

    print("  Executor is processing the batch...")
    for seq in running_batch:
        seq.output_tokens.append(0) # Add a dummy token
        # Check if a sequence is finished
        if len(seq) >= MAX_TOKENS_PER_SEQ:
            seq.status = SequenceStatus.FINISHED
            print(f"    - Sequence {seq.seq_id} has finished.")
    print("")

### Going Deeper: Preemption and Fairness

Our simple scheduler uses a First-In, First-Out (FIFO) policy. It schedules requests in the order they arrive. What's a potential problem with this?

Imagine `req_A` is a very long research query that will take thousands of tokens to generate. Right behind it in the queue is `req_B`, a simple question that only needs 20 tokens. If `req_A` gets scheduled, it could occupy GPU resources for a long time, making the user who submitted `req_B` wait unnecessarily. This is called **head-of-line blocking**.

To solve this, real-world schedulers like vLLM's implement **preemption**.

The idea is simple: if a higher-priority request is waiting and there isn't enough memory, the scheduler can decide to "preempt" (pause) a lower-priority running request.
1.  **Swap Out:** The preempted request's state is changed from `RUNNING` back to `WAITING`. Its KV cache blocks are freed and copied to CPU RAM.
2.  **Swap In:** The freed blocks are now given to the high-priority request, which can start running immediately.
3.  **Resume:** Later, when resources become available again, the preempted request can be swapped back in from CPU to GPU to resume its generation.

This ensures that short, quick jobs can jump the queue, improving average latency and making the system feel more responsive and fair to all users. While implementing the swap mechanism is complex, the core decision logic happens right inside the scheduler.

In [ ]:
# === Visualization ===
# Let's visualize the flow of requests and memory usage over time.
# This ASCII art will make the dynamics of continuous batching clear.

def visualize_state(step, scheduler, total_blocks):
    """Helper function to print the system state."""
    waiting_ids = [s.seq_id for s in scheduler.waiting_queue]
    running_ids = [s.seq_id for s in scheduler.running_sequences.keys()]
    
    num_used_blocks = total_blocks - len(scheduler.kv_cache.free_blocks)
    usage_bar = '█' * num_used_blocks + '░' * (total_blocks - num_used_blocks)
    
    print(f"Step {step:<2} | "
          f"Waiting: {str(waiting_ids):<15} | "
          f"Running: {str(running_ids):<15} | "
          f"KV Cache: [{usage_bar}] ({num_used_blocks}/{total_blocks})")

# --- Reset and run a new simulation for visualization ---
NUM_BLOCKS = 10
MAX_LEN = 15 # Shorter sequences for a quicker demo
kv_cache = KVCacheManager(num_total_blocks=NUM_BLOCKS)
scheduler = Scheduler(kv_cache)

# Different requests to show swapping
requests = [
    Sequence(seq_id="A_long", prompt_tokens=[1]*35),  # 4 blocks
    Sequence(seq_id="B_med", prompt_tokens=[2]*25),   # 3 blocks
    Sequence(seq_id="C_med", prompt_tokens=[3]*25),   # 3 blocks
    Sequence(seq_id="D_short", prompt_tokens=[4]*5), # 1 block
]
for req in requests:
    scheduler.add_sequence(req)

print("--- Visualization of Continuous Batching ---\n")
visualize_state(0, scheduler, NUM_BLOCKS)

# Simulation loop
for step in range(1, 15):
    running_batch = scheduler.schedule()

    # Simulate executor work and finishing sequences
    for seq in running_batch:
        seq.output_tokens.append(0)
        # Randomly finish sequences to show dynamic freeing
        if len(seq) >= MAX_LEN or (seq.seq_id == "D_short" and len(seq) > 8):
             seq.status = SequenceStatus.FINISHED
    
    visualize_state(step, scheduler, NUM_BLOCKS)

    if not scheduler.running_sequences and not scheduler.waiting_queue:
        print("\n--- Simulation Complete ---")
        break

### Summary

In this notebook, we demystified the core of vLLM's high throughput: the request scheduler and its use of continuous batching.

**What We Built:**
We implemented a simplified `Scheduler` that interacts with a `KVCacheManager`. This scheduler successfully manages the lifecycle of multiple requests, moving them from a waiting queue to a running state as memory becomes available, and forming them into batches for processing.

**Key Takeaways:**
*   **Continuous Batching is Dynamic:** Unlike static batching, it makes decisions at every single step, ensuring the GPU is never idle if there's work to be done.
*   **The Scheduler is the Conductor:** It orchestrates the complex dance between incoming requests, running sequences, and finite memory resources (the KV cache).
*   **Memory is the Bottleneck:** The primary constraint in scheduling is the availability of KV cache blocks. The ability to allocate memory for a request is the gatekeeper that decides whether it runs now or waits.

**What's Next:**
We have now built simplified versions of the `Executor` (which runs the model) and the `Scheduler` (which decides what to run). The next notebook, **Orchestrating the Inference Engine Core**, will show how these two components are brought together by the `EngineCore` to create the complete, powerful, and efficient inference loop that powers vLLM.